# District logistics demo — Pune, Maharashtra

End-to-end example combining every dataset class in this repo: administrative boundaries, roads by class, rail lines and stations, logistics hubs, and Census 2011 demographics.

Prerequisite (network needed, ~1-2 min): `python scripts/make_demo.py --district Pune` which writes `examples/output/pune_logistics.gpkg`. This notebook reads that GeoPackage plus the committed datasets.

In [ ]:
import pathlib
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

REPO = pathlib.Path.cwd().parent
gpkg = REPO / "examples" / "output" / "pune_logistics.gpkg"
assert gpkg.exists(), "Run scripts/make_demo.py --district Pune first"
print(list(gpd.list_layers(gpkg).name))

## 1. Layers in the demo GeoPackage

In [ ]:
boundary = gpd.read_file(gpkg, layer="district_boundary")
roads = gpd.read_file(gpkg, layer="roads")
rail = gpd.read_file(gpkg, layer="rail_lines")
stations = gpd.read_file(gpkg, layer="rail_stations")
print(f"road segments: {len(roads)}, by class:
{roads.road_class.value_counts()}")
print(f"
rail segments: {len(rail)}, stations: {len(stations)}")

## 2. Demographics join (LGD/census keys)

In [ ]:
census = pd.read_csv(REPO / "data" / "demographic" / "census2011_district_key_indicators.csv")
row = census[census.district.str.lower() == "pune"].iloc[0]
pct = row.Literate / row.Population
print(f"Pune 2011: population {row.Population:,.0f}, literate {row.Literate:,.0f} ({pct:.1%}), workers {row.Workers:,.0f}")

## 3. Multi-layer map

In [ ]:
fig, ax = plt.subplots(figsize=(11, 11))
boundary.boundary.plot(ax=ax, color="#2c3e50", linewidth=1.2)
styles = {"NH": ("#d62728", 1.6), "SH": ("#ff7f0e", 1.1),
          "MDR": ("#bcbd22", 0.7), "ODR": ("#7f7f7f", 0.4)}
for cls, (c, w) in styles.items():
    sub = roads[roads.road_class == cls]
    if len(sub):
        sub.plot(ax=ax, color=c, linewidth=w, label=f"Road {cls}")
if len(rail):
    rail.plot(ax=ax, color="#17a2b8", linewidth=0.8, linestyle="--", label="Railway")
if len(stations):
    stations.plot(ax=ax, color="k", markersize=3, marker="s", label="Stations")
hubs_all = pd.concat([pd.read_csv(REPO / "data" / "logistics_hubs" / f)
                      for f in ("ports.csv", "icds.csv", "icps.csv", "air_cargo.csv")])
hubs = gpd.GeoDataFrame(hubs_all,
    geometry=gpd.points_from_xy(hubs_all.longitude, hubs_all.latitude), crs=4326)
hubs = hubs[hubs.within(boundary.unary_union)]
if len(hubs):
    hubs.plot(ax=ax, color="#8e44ad", markersize=60, marker="^", edgecolor="w", label="Hubs")
ax.legend(loc="lower left", fontsize=9)
ax.set_axis_off()
plt.figtext(0.99, 0.01, "Boundaries indicative (DataMeet/OSM) — not authoritative "
            "depictions; © OpenStreetMap contributors (ODbL)",
            ha="right", fontsize=6, color="#555555")
plt.show()

## 4. What this demonstrates
- Boundaries, roads, rail, hubs all in EPSG:4326 and joinable via state/district names and LGD codes.
- Road classification from OSM tags into NH/SH/MDR/ODR.
- The same pattern works for any district: `make_demo.py --district <name> --state <name>`.